# Action Castle, Multi-Agent Edition

This notebook rebuilds **Action Castle** on top of the new agent framework, with every
NPC driven by an *agent* — an LLM-shaped decision-maker — instead of a hand-written
script. It runs **fully offline**: the "LLM" is a `MockLlmClient`, so the whole thing is
deterministic, free, and needs no API key.

It exercises the framework features we've merged so far, in one playable game:

| Feature | Issue | What you'll see here |
|---|---|---|
| Mock LLM client | #2 | `MockLlmClient` stands in for a real LLM, scripted *and* prompt-reactive |
| Agent decision seam | #3 | `LLMAgent.decide(observation) → command`, wired to NPCs via `make_react_behavior` |
| Event log + triggers | #6 | `game.events`, `game.add_trigger(...)`, `game.schedule_event(...)` |
| Time model | #7 | An in-game clock: `time_config`, day periods, time shown in observations |

The player's commands are scripted (this is a demo, not an interactive session), but the
NPCs are **not** scripted in the old sense: each one perceives the world through the same
observation prompt a real LLM would receive, decides on a raw command string, and that
command is routed through the parser's `check_preconditions()` gate like everyone else's.

Everything here runs in the engine's default **sequential** turn mode: the player
acts, then each NPC observes the *already-changed* world and acts, one after another.
A companion notebook, **`06_simultaneous_turns.ipynb`**, demonstrates the
opt-in **simultaneous** turn mode (issue #25).

> **Run this from the `notebooks/` directory** (the Jupyter kernel's working directory
> must contain `hw1_solution/`), with the repo installed: `pip install -e ".[dev]"`.


## 1. The agent seam: `decide(observation) → command`

The core abstraction from issue #3 lives in `text_adventure_games/npc.py`. An `Agent`
owns a character's *mind* — its persona and goals — and exposes exactly one decision
method:

```
Agent.decide(observation: str) -> command: str | None
```

`LLMAgent` implements the seam by asking an LLM; `ScriptedAgent` implements it with a
plain Python rule. Because both sit behind the same seam, **the game can't tell which
backend is driving a character** — and neither can the tests.

The pieces *around* the seam (building the observation, routing the command through the
parser, retrying on failure) live in `react_behavior`, and `make_react_behavior` bridges
the whole loop onto a character via `Character.set_behavior()`.

First, the seam in isolation. A queue-style `MockLlmClient` (issue #2) returns canned
responses one per call — note how the persona and goals end up in the system message,
exactly as a real provider would receive them:


In [1]:
import time

from rich.console import Console

from text_adventure_games import games, things
from text_adventure_games.llm_client import MockLlmClient
from text_adventure_games.npc import LLMAgent, make_react_behavior, build_npc_context
from text_adventure_games.reporting import NORMAL, RichTerminalRenderer
from text_adventure_games.things.characters import Goal, GoalType
from text_adventure_games.triggers import has_property, in_location

# Game content (custom actions and blocks) reused from the HW1 solution.
from hw1_solution.action_castle import (
    # player-facing custom actions
    Unlock_Door, Read_Runes, Propose, Wear_Crown, Sit_On_Throne,
    # NPC social actions
    Growl, Snarl, Pound_Fists, Warn, Threaten, Haunt, Ghost_Touch,
    # blocks
    Troll_Block, Guard_Block, Darkness, Door_Block,
)


def force_rich_notebook_output(game, level=NORMAL):
    """Use Rich's Jupyter renderer for parser output in this notebook."""
    game.parser.set_renderer(
        RichTerminalRenderer(
            level=level,
            console=Console(force_jupyter=True, width=100),
        )
    )
    return game

In [2]:
demo_client = MockLlmClient(["growl", "snarl"])
demo_agent = LLMAgent(
    demo_client,
    persona="I am a hungry troll.",
    goals=[Goal("guard the bridge", GoalType.MEDIUM)],
)

print("decide() returned:", repr(demo_agent.decide("The player approaches the bridge.")))
print()
print("--- what the 'LLM' was actually sent (system message) ---")
print(demo_client.calls[0]["messages"][0]["content"])

decide() returned: 'growl'

--- what the 'LLM' was actually sent (system message) ---
You are an NPC in a text adventure game.
Persona: I am a hungry troll.
Goals:
Medium-term:
  - guard the bridge
Based on your persona, goals, and the current situation, choose a single game command to execute. Reply with exactly three lines:
Reasoning: <one short sentence explaining your choice>
Action: <the command, e.g. 'attack player', 'go north', 'take sword'>
Duration: <estimated in-game minutes this action takes, e.g. 5>


## 2. Writing the NPC brains

A queue of canned responses is fine for a unit test, but an NPC needs to *react*.
`MockLlmClient` also accepts a **callable** `(messages, max_tokens, temperature) → str | None`
— it gets the exact prompt a real LLM would get, and returns whatever "the model" says.

So each brain below is a tiny fake LLM. The rules of the game (for us, not the NPCs):

- **The brain only sees the prompt.** No peeking at game objects. If a fact isn't in the
  observation, the agent cannot act on it — exactly the constraint a real LLM operates
  under. (You'll see one consequence below: the observation doesn't include the troll's
  own `is_hungry` property, so its brain has to *infer* it was fed from the recent-events
  history. Richer observations are future work — issue #9.)
- **Returning `None` means "do nothing this turn"** — the agent layer skips the parser
  entirely.
- Commands are returned **without the character's name** — the behavior bridge prepends
  it before parsing (`"growl"` becomes `"troll growl"`), and the parser's precondition
  gate takes it from there.

One more thing to notice: the brain factories below return closures that **carry state**
(how many threats has the troll made? has the princess already proposed?). That state is
consumed as a game plays, which is why §5 mints a *fresh* brain for every new game.

A couple of helpers for picking the observation apart:


In [3]:
def observation_of(messages):
    """The observation is the last user message the agent was sent."""
    return messages[-1]["content"]


def scene_of(messages):
    """Just the here-and-now part of the observation (drop the history)."""
    return observation_of(messages).split("Recent events:")[0]


def player_is_here(messages):
    """Characters present are listed as ' * <name> - <description>' lines."""
    return "* The player" in scene_of(messages)

In [4]:
def make_troll_brain():
    """Escalates while the player loiters; settles down once it has been fed."""
    threats = ["growl", "snarl", "attack with club"]
    state = {"step": 0, "fed": False}

    def brain(messages, max_tokens, temperature):
        # The troll can't see its own is_hungry property in the observation,
        # so it infers "I was fed" from the recent-events history.
        recent = observation_of(messages).lower()
        if "gave the fish to troll" in recent or "troll eats the fish" in recent:
            state["fed"] = True
        if state["fed"] or not player_is_here(messages):
            state["step"] = 0
            return None
        command = threats[min(state["step"], len(threats) - 1)]
        state["step"] += 1
        return command

    return brain

# Checking if troll is fed via string could cause issues
# Look at actions --> preconditions + effects (which affect the world state)


def make_guard_brain():
    """Warns once, threatens once, then it's swords."""
    script = ["warn", "threaten", "attack with sword"]
    state = {"step": 0}

    def brain(messages, max_tokens, temperature):
        if not player_is_here(messages):
            state["step"] = 0
            return None
        command = script[min(state["step"], len(script) - 1)]
        state["step"] += 1
        return command

    return brain

In [5]:
def make_ghost_brain():
    """One chilling warning, then the icy hand. Banish it before turn two."""
    state = {"turns_seen": 0}

    def brain(messages, max_tokens, temperature):
        if not player_is_here(messages):
            return None
        state["turns_seen"] += 1
        if state["turns_seen"] == 1:
            return "haunt"
        return "ghost ghost touch the player"

    return brain


def make_princess_brain():
    """Waits in the tower. If she's holding the rose, she knows what it means."""
    state = {"proposed": False}

    def brain(messages, max_tokens, temperature):
        if state["proposed"] or not player_is_here(messages):
            return None
        if "* rose" in scene_of(messages):  # the rose shows up in her inventory
            state["proposed"] = True
            return "propose marriage"
        return None

    return brain

## 3. Build the world

The classic Action Castle map. `add_connection` automatically creates the reverse
connection for cardinal directions, `up`/`down`, and `in`/`out`.

Note the shape of everything from here on: **builder functions**, not top-level objects.
Playing a game *mutates* the world — items move, characters get knocked out, the door
gets unlocked, someone ends up on the throne — so a played-out world can't host a second
playthrough. Builders let `build_game()` (§5) mint a brand-new world for every run.


In [6]:
def build_world():
    """Create all locations (with their items) and connect them. Fresh every call."""
    cottage = things.Location("Cottage", "You are standing in a small cottage.")
    garden_path = things.Location(
        "Garden Path", "You are standing on a lush garden path. There is a cottage here."
    )
    fishing_pond = things.Location(
        "Fishing Pond", "You are at the edge of a small fishing pond."
    )
    winding_path = things.Location(
        "Winding Path", "You are walking along a winding path. There is a tall tree here."
    )
    top_of_tree = things.Location(
        "Top of the Tall Tree", "You are at the top of the tall tree."
    )
    drawbridge = things.Location(
        "Drawbridge",
        "You are standing on one side of a drawbridge leading to ACTION CASTLE.",
    )
    courtyard = things.Location("Courtyard", "You are in the courtyard of ACTION CASTLE.")
    tower_stairs = things.Location(
        "Tower Stairs",
        "You are climbing the stairs to the tower. There is a locked door here.",
    )
    tower = things.Location("Tower", "You are inside a tower.")
    dungeon_stairs = things.Location(
        "Dungeon Stairs", "You are climbing the stairs down to the dungeon."
    )
    dungeon = things.Location(
        "Dungeon", "You are in the dungeon. There is a spooky ghost here."
    )
    feasting_hall = things.Location(
        "Great Feasting Hall", "You stand inside the Great Feasting Hall."
    )
    throne_room = things.Location(
        "Throne Room", "This is the throne room of ACTION CASTLE."
    )

    cottage.add_connection("out", garden_path)
    garden_path.add_connection("south", fishing_pond)
    garden_path.add_connection("north", winding_path)
    winding_path.add_connection("up", top_of_tree)
    winding_path.add_connection("east", drawbridge)
    drawbridge.add_connection("east", courtyard)
    courtyard.add_connection("up", tower_stairs)
    tower_stairs.add_connection("up", tower)
    courtyard.add_connection("down", dungeon_stairs)
    dungeon_stairs.add_connection("down", dungeon)
    courtyard.add_connection("east", feasting_hall)
    feasting_hall.add_connection("east", throne_room)

    fishing_pole = things.Item("pole", "a fishing pole", "A SIMPLE FISHING POLE.")
    cottage.add_item(fishing_pole)

    branch = things.Item("branch", "a stout, dead branch", "IT LOOKS LIKE A GOOD CLUB.")
    branch.set_property("is_weapon", True)
    branch.set_property("is_fragile", True)
    top_of_tree.add_item(branch)

    candle = things.Item(
        "candle", "a strange candle", "THE CANDLE IS COVERED IN STRANGE RUNES."
    )
    candle.set_property("flammable", True)
    candle.add_command_hint("light candle")
    candle.add_command_hint("read runes")
    feasting_hall.add_item(candle)

    pond = things.Item("pond", "a small fishing pond", "THERE ARE FISH IN THE POND.")
    pond.set_property("gettable", False)
    pond.set_property("has_fish", True)
    pond.add_command_hint("catch fish with pole")
    fishing_pond.add_item(pond)
    fishing_pond.set_property("has_fish", True)

    rosebush = things.Item(
        "rosebush", "a rosebush", "THE ROSEBUSH CONTAINS A SINGLE RED ROSE."
    )
    rosebush.set_property("gettable", False)
    rosebush.set_property("has_rose", True)
    rosebush.add_command_hint("pick rose")
    garden_path.add_item(rosebush)

    throne = things.Item("throne", "An ornate golden throne.")
    throne.set_property("gettable", False)
    throne.add_command_hint("sit on throne")
    throne_room.add_item(throne)

    door = things.Item("door", "a door", "THE DOOR IS SECURELY LOCKED.")
    door.set_property("gettable", False)
    door.set_property("is_locked", True)
    door.add_command_hint("unlock door")
    tower_stairs.add_item(door)

    return {
        "cottage": cottage,
        "garden_path": garden_path,
        "fishing_pond": fishing_pond,
        "winding_path": winding_path,
        "top_of_tree": top_of_tree,
        "drawbridge": drawbridge,
        "courtyard": courtyard,
        "tower_stairs": tower_stairs,
        "tower": tower,
        "dungeon_stairs": dungeon_stairs,
        "dungeon": dungeon,
        "feasting_hall": feasting_hall,
        "throne_room": throne_room,
    }

## 4. The cast

Four NPCs, each with a persona (their character) and a brain (their decision-maker).
A character's goals live on the *character* now (issue #23) — tiered short/medium/long,
and re-read every turn so an in-game `add_goal()` lands in the next decision. So the
wiring is: attach the goals, then give the character a brain.

```python
character.add_goal("guard the bridge", GoalType.MEDIUM)
character.set_behavior(make_react_behavior(llm_client))
```

`make_react_behavior` builds an `LLMAgent` around the client and runs the
Observe → Decide → Act (→ Reflect on failure) loop on the character's turn. Swapping a
mock brain for a real model is *only* a change of client — the rest of this notebook
would run unmodified.

In [7]:
def build_cast(locations):
    """Create the four NPCs with freshly minted brains; place them in the world.

    Returns (cast, clients): the characters and their MockLlmClients, both
    keyed by name. We keep the clients so we can inspect their call logs later.
    """
    troll = things.Character(
        name="troll",
        description="A mean troll",
        persona="I am hungry. I guard the drawbridge and keep strangers out of the castle.",
    )
    troll.set_property("is_hungry", True)
    club = things.Item("club", "a heavy club", "A CRUDE BUT DEADLY WEAPON.")
    club.set_property("is_weapon", True)
    troll.add_to_inventory(club)
    locations["drawbridge"].add_character(troll)

    guard = things.Character(
        name="guard",
        description="A castle guard",
        persona="I am suspicious of anyone trying to enter the castle.",
    )
    guard.set_property("emotional_state", "suspicious")
    key = things.Item("key", "a brass key", "THIS LOOKS USEFUL.")
    guard.add_to_inventory(key)
    sword = things.Item("sword", "a short sword", "A SHARP SHORT SWORD.")
    sword.set_property("is_weapon", True)
    guard.add_to_inventory(sword)
    locations["courtyard"].add_character(guard)

    princess = things.Character(
        name="princess",
        description="A beautiful princess in a peaked hat",
        persona="I am the princess. I am grieving my father's death and I am lonely.",
    )
    princess.set_property("is_royal", True)
    princess.set_property("emotional_state", "sad and lonely")
    locations["tower"].add_character(princess)

    ghost = things.Character(
        name="ghost",
        description="A ghost with bony, claw-like fingers, wearing a crown",
        persona="I was murdered. I haunt this castle until I am banished.",
    )
    ghost.set_property("is_undead", True)
    crown = things.Item("crown", "a crown", "A CROWN FIT FOR A KING.")
    crown.set_property("wearable", True)  # affordance tag (#53); without it 'wear crown' is blocked
    crown.add_command_hint("wear crown")
    ghost.add_to_inventory(crown)
    locations["dungeon"].add_character(ghost)

    # One client per NPC, each wrapping a brand-new brain closure.
    clients = {
        "troll": MockLlmClient(make_troll_brain()),
        "guard": MockLlmClient(make_guard_brain()),
        "ghost": MockLlmClient(make_ghost_brain()),
        "princess": MockLlmClient(make_princess_brain()),
    }

    # Goals live on the character now (issue #23), tiered short/medium/long;
    # the behavior re-reads them off the character each turn. Attach the goals,
    # then wire the brain.
    troll.add_goal("eat", GoalType.SHORT)
    troll.add_goal("guard the bridge", GoalType.MEDIUM)
    troll.set_behavior(make_react_behavior(clients["troll"]))

    guard.add_goal("keep strangers out", GoalType.MEDIUM)
    guard.set_behavior(make_react_behavior(clients["guard"]))

    ghost.add_goal("scare intruders away", GoalType.SHORT)
    ghost.set_behavior(make_react_behavior(clients["ghost"]))

    princess.add_goal("find true love", GoalType.LONG)
    princess.set_behavior(make_react_behavior(clients["princess"]))

    cast = {"troll": troll, "guard": guard, "princess": princess, "ghost": ghost}
    return cast, clients

In [8]:
def build_player():
    """The hero: a simple peasant with a lamp."""
    player = things.Character(
        name="The player",
        description="You are a simple peasant destined for greatness.",
        persona="I am on an adventure.",
    )
    lamp = things.Item("lamp", "a lamp", "A LAMP.")
    lamp.set_property("flammable", True)
    lamp.add_command_hint("light lamp")
    player.add_to_inventory(lamp)
    return player

# Character's goals: short, medium, long term
# Persona is (not totally) immutable -- doesnt change frequently, goal dynamic


def add_blocks(locations, cast):
    """The puzzles: blocks gate movement until their condition is met."""
    locations["drawbridge"].add_block(
        "east", Troll_Block(locations["drawbridge"], cast["troll"])
    )
    locations["courtyard"].add_block(
        "east", Guard_Block(locations["courtyard"], cast["guard"])
    )
    locations["dungeon_stairs"].add_block("down", Darkness(locations["dungeon_stairs"]))
    door = locations["tower_stairs"].items["door"]
    locations["tower_stairs"].add_block("up", Door_Block(locations["tower_stairs"], door))

## 5. Assemble the game — with a clock

A new game is a `Game` subclass with a win condition (`is_won`). New in issue #7:
passing `time_config` gives the game a `GameClock` that maps the turn counter onto
in-game time of day. Time is **opt-in** and **stateless** — the clock is a pure function
of `game.turn`, so it can never drift out of sync with the loop.

We start at 8:00 AM with 20 in-game minutes per turn, so our adventure will play out
over a single day — watch the day periods (morning → afternoon → dusk → night) change in
the transcript.


In [9]:
class ActionCastleV2(games.Game):
    def is_won(self) -> bool:
        for character in self.characters.values():
            if character.get_property("is_reigning"):
                self.game_over_description = (
                    f"{character.name.title()} now reigns over ACTION CASTLE. THE END."
                )
                return True
        return False

## 6. Triggers and scheduled events

Issue #6 added a **react phase** at the end of every round: after the player and all
NPCs have acted, the game fires any `Trigger` whose condition is now true, and records
everything in the event log. Three flavors here:

1. a **scheduled event** — `schedule_event(turn, callback)` is sugar for a one-shot
   `at_turn` trigger;
2. a **property trigger** — fires the moment the troll stops being hungry;
3. a **location trigger** — fires when the player first reaches the throne room.

(There's also `every(n)` for recurring conditions, and `all_of`/`any_of` for compound
ones.)


In [10]:
def add_story_triggers(game):
    """Register the three story triggers on a freshly built game."""
    troll = game.characters["troll"]
    throne_room = game.locations["Throne Room"]

    def bells(g):
        g.parser.ok("Far away, the castle bells ring out the late-morning hour.")

    def troll_fed_narration(g):
        g.parser.ok(
            "Its belly full at last, the troll slumps against the drawbridge and dozes off."
        )

    def fanfare(g):
        g.parser.ok("Unseen trumpeters strike up a fanfare as you enter the throne room.")

    game.schedule_event(10, bells, name="castle-bells")
    game.add_trigger(
        "troll-fed", has_property(troll, "is_hungry", False), troll_fed_narration
    )
    game.add_trigger(
        "throne-room-fanfare", in_location(game.player, throne_room), fanfare
    )

### Putting it together: `build_game()`

One function mints a brand-new game: fresh world, fresh cast, **freshly minted brains**
(remember, they carry closure state — a princess who has already proposed would never
propose again), armed triggers, and a clock set to 8:00 AM. Call it as many times as you
like; every call is a clean slate. This is what makes the play cell in §8 safely
re-runnable.


In [11]:
def build_game():
    """A brand-new ActionCastleV2, plus the NPCs' mock clients for inspection."""
    locations = build_world()
    cast, clients = build_cast(locations)
    player = build_player()
    add_blocks(locations, cast)

    game = ActionCastleV2(
        locations["cottage"],
        player,
        characters=list(cast.values()),
        custom_actions=[
            Unlock_Door, Read_Runes, Propose, Wear_Crown, Sit_On_Throne,
            Growl, Snarl, Pound_Fists, Warn, Threaten, Haunt, Ghost_Touch,
        ],
        time_config={"start_hour": 8, "minutes_per_turn": 20},
    )
    force_rich_notebook_output(game)
    add_story_triggers(game)
    return game, clients


game, clients = build_game()
print("The game begins at:", game.current_time())
print("Triggers armed:", [t.name for t in game.triggers])

# LLM to estimate the length of the game

The game begins at: 8:00 AM (morning)
Triggers armed: ['castle-bells', 'troll-fed', 'throne-room-fanfare']


## 7. What an agent actually sees

Before playing, it's worth looking at the observation prompt an agent receives on its
turn — this string is the agent's *entire* knowledge of the world. `build_npc_context`
combines `game.describe_for(character)` (location, exits, items, other characters, its
own inventory, the available actions, and the turn/time) with recent history:


In [12]:
print(build_npc_context(game.characters["troll"], game))

DRAWBRIDGE
You are standing on one side of a drawbridge leading to ACTION CASTLE.
Exits:
 * West to Winding Path
 * East to Courtyard
Inventory:
 * club - a heavy club [gettable]
Available actions: adopt goal, attack, catch fish, describe, drink, drop, drop goal, eat, examine, get, ghost touch, give, go, growl, haunt, inventory, light, pick rose, pound fists, propose, quit, read runes, say, sequence, sit on throne, smell rose, snarl, take off, threaten, unlock door, unwield, wait, warn, wear, wear crown, wield
Turn: 0 (8:00 AM (morning))


## 8. Play!

The player's commands are scripted below; everything the NPCs do is decided live by
their agents, turn by turn. `echo_commands = True` makes the parser print every command
it routes — including the NPCs' — so you can watch the `decide() → parser` pipeline work.

`run_game` **streams the transcript**: one player command per round, with a short pause
between rounds so you can read along (the full story takes about three minutes at the
default pause — pass `pause=0` for an instant transcript, or a bigger number to savor
it). And because the
cell starts with `build_game()`, **every run plays a brand-new world** — re-run it as
often as you like.

Things to watch for:

- **turn 11–13** — the troll growls the moment we arrive; we dawdle (`look`) and it
  escalates to a snarl; we hand over the fish and the **troll-fed trigger** narrates the
  result. Had we lingered two more turns, `attack with club` was next.
- **turn 10** — the castle bells (scheduled event) ring, mid-journey.
- **turn 14–15** — the guard warns us *once*; the branch settles the discussion.
- **turn 23–24** — the ghost gets exactly one haunt in before the runes banish it. One
  turn slower and its icy hand stops our heart.
- **turn 30–32** — we hand the princess the rose. On *her* turn, *she* proposes to *us*
  — agent-initiated interaction, parsed and precondition-checked like any player command
  (`Propose` requires both parties present, happy, and unmarried).


In [13]:
WALKTHROUGH = [
    "take pole",
    "go out",
    "pick rose",
    "smell rose",             # happiness is a precondition for marriage!
    "go south",
    "catch fish with pole",
    "go north",
    "go north",
    "go up",
    "take branch",
    "go down",
    "go east",                # drawbridge: the troll growls
    "look",                   # dawdle: the troll escalates to a snarl
    "give fish to troll",     # it devours the fish; the troll-fed trigger fires
    "go east",                # courtyard: the guard warns us
    "hit guard with branch",  # ...we don't wait for the sword
    "take key",
    "go east",                # feasting hall
    "take candle",
    "light candle",
    "go west",
    "light lamp",
    "go down",
    "go down",                # dungeon: the ghost haunts us
    "read runes",             # banish it before the icy touch
    "take crown",
    "go up",
    "go up",
    "go up",
    "unlock door",
    "go up",                  # tower: the princess waits
    "give rose to princess",  # she smells it... and decides something
    "wear crown",
    "go down",
    "go down",
    "go east",
    "go east",                # throne room: fanfare trigger
    "sit on throne",
]

In [14]:
def run_game(game, commands, pause=5.0):
    """Play scripted player commands against a FRESH game, streaming the
    transcript with `pause` seconds between rounds."""
    if game.turn != 0 or game.is_game_over():
        raise RuntimeError(
            "This game instance has already been played — "
            "call build_game() for a fresh one."
        )
    game.parser.echo_commands = True
    game.parser.parse_command("look")
    for command in commands:
        time.sleep(pause)
        print(f"\n[{game.current_time()} | turn {game.turn}]", flush=True)
        success = game.do_command(command)
        if not success:
            print(f"!!! '{command}' failed — the world is out of sync with the script.")
            print("!!! Re-run this cell: build_game() will mint a fresh world.")
            break
        if game.is_game_over():
            print("\n" + "*" * 60, flush=True)
            print(game.game_over_description)
            print("*" * 60)
            break


game, clients = build_game()  # a brand-new world, every run
run_game(game, WALKTHROUGH)

[player command] look

[narration] COTTAGE
            (8:00 AM (morning))
            You are standing in a small cottage.
            Exits:
             * Out to Garden Path
            
            You see:
             * pole - a fishing pole
            
            


[8:00 AM (morning) | turn 0]


[player command] take pole

[narration] The player got the pole.

...


[8:20 PM (night) | turn 37]


[player command] sit on throne

[narration] The Player now sits upon the throne. The reign of The Player has begun!


************************************************************


The Player now reigns over ACTION CASTLE. THE END.
************************************************************


## 9. The paper trail

Every successful action — player, NPC, or trigger — was recorded in the event log
(issue #6) as a `GameEvent` with the turn it happened on. This is the raw material for
agent memory (what did I see happen?), for analysis, and for debugging multi-agent
runs.


In [15]:
print(f"{len(game.events)} events were logged:\n")
for event in game.events:
    print(f"  turn {event.turn:>2}  {event.actor:<12} {event.action:<14} {event.summary}")

47 events were logged:

  turn  0  The player   describe       look
  turn  0  The player   get            take pole
  turn  1  The player   go             go out
  turn  2  The player   pick rose      pick rose
  turn  3  The player   smell rose     smell rose
  turn  4  The player   go             go south
  turn  5  The player   catch fish     catch fish with pole
  turn  6  The player   go             go north
  turn  7  The player   go             go north
  turn  8  The player   go             go up
  turn  9  The player   get            take branch
  turn 10  trigger      castle-bells   castle-bells fired
  turn 10  The player   go             go down
  turn 11  The player   go             go east
  turn 12  troll        growl          growl
  turn 12  The player   describe       look
  turn 13  troll        snarl          snarl
  turn 13  The player   give           give fish to troll
  turn 14  trigger      troll-fed      troll-fed fired
  turn 14  The player   go             

In [16]:
for name, client in clients.items():
    print(f"The {name}'s brain was consulted {len(client.calls)} times.")

The troll's brain was consulted 38 times.
The guard's brain was consulted 15 times.
The ghost's brain was consulted 24 times.
The princess's brain was consulted 37 times.


Those call counts are worth a pause: **every living NPC's agent is consulted every
round**, whether or not anything interesting is happening near it. With a mock that's
free; with a real LLM that's ~115 API calls for one playthrough. Cost control (skip
far-away NPCs, batch calls, use small models) is part of the agent-loop design work
ahead.

## 10. Swapping in a real LLM

Because everything runs through the `decide()` seam, upgrading an NPC from mock to real
is a one-line change per character (the troll keeps the goals `build_cast` already gave
it — the behavior reads them off the character):

```python
from text_adventure_games.llm_client import LlmConfig, create_llm_client

llm = create_llm_client(LlmConfig(provider="anthropic"))   # needs ANTHROPIC_API_KEY
game, clients = build_game()
game.characters["troll"].set_behavior(make_react_behavior(llm))
```

There's also `make_hybrid_behavior(llm, scripted_fallback)` — try the LLM, fall back to
a scripted behavior if it errors — which is what the web app uses when `LLM_PROVIDER`
is set.

### Known warts (good first fixes!)

- **Observations omit the agent's own state:** the troll can't see its own `is_hungry`
  property, so its brain infers it from event history. `describe_for()` should
  probably include the character's own properties (and feed into issue #9's generated
  descriptions).
- **`Character.take_turn` is all-or-nothing:** agents whose brains return `None` still
  paid an LLM round-trip in a real deployment (see the call counts above).

### Where this goes next

- **Issue #4** — a real *Reflect* step in the ReAct loop (today a failed command just
  gets one mechanical retry with the failure appended).
- **Issue #8** — agent-to-agent interaction: conversations, trades, alliances. The
  princess proposing is a first taste.
- **Issue #9** — richer, LLM-generated observations and descriptions, time- and
  property-aware.